# Support Vector Regression

There are three different implementations of Support Vector Regression:
SVR, NuSVR and LinearSVR.

LinearSVR provides a faster implementation than SVR but only considers linear kernels,
while NuSVR implements a slightly different formulation than SVR and LinearSVR.

See Implementation details for further details.


In [1]:
from sklearn.datasets import fetch_california_housing
california_housing = fetch_california_housing(as_frame=True)

#Creating feature and target arrays
X = california_housing.data
y = california_housing.target
columns = california_housing.feature_names

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(copy=False).fit(X)
scaler.transform(X)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X,y,
                                                    test_size=0.3,
                                                    random_state=42)

## The Optimization Problem

$$
\begin{align}\begin{aligned}\min_ {w, b, \zeta, \zeta^*} \frac{1}{2} ||w||^2 + C \sum_{i=1}^{n} (\zeta_i + \zeta_i^*)\\\begin{split}\textrm {subject to } & y_i - (w^T \phi (x_i) + b) \leq \varepsilon + \zeta_i,\\
                      & (w^T \phi (x_i) + b) - y_i \leq \varepsilon + \zeta_i^*,\\
                      & \zeta_i, \zeta_i^* \geq 0, i=1, ..., n\end{split}\end{aligned}\end{align}
$$

where $\phi$ is the kernel map that can be chosen as
 - linear: $\langle x, x'\rangle$
 - poly: $(\gamma \langle x, x'\rangle + r)^d$ where $\gamma,\, d ,\, r$ are specified by keyword gamma, degree and coef0 respectively.
 - rbf - radial basis function: $\exp(-\gamma \|x-x'\|^2)$
 - neural networks (sigmoid): $\tanh(\gamma \langle x,x'\rangle + r)$

## Grid Search

C (Parámetro de Regularización): Este parámetro controla el equilibrio entre lograr un error bajo en los datos de entrenamiento y minimizar la norma de los pesos. Un C más pequeño enfatiza un margen más grande y tolera más errores de entrenamiento, mientras que un C más grande busca menos errores de entrenamiento, pero podría conducir a un margen más pequeño. Esencialmente, dicta cuánto quieres penalizar los errores.

epsilon (Tubo Epsilon): En SVR, epsilon define un margen de tolerancia donde no se asocia ninguna penalización con los errores que caen dentro de este margen. Los puntos de datos dentro de la distancia epsilon de la predicción no son penalizados. Esto significa que las predicciones que están a una distancia epsilon del valor real se consideran correctas. Un epsilon más grande significa una mayor tolerancia para los errores.

gamma (Coeficiente del Kernel): Este parámetro define qué tan lejos llega la influencia de un único ejemplo de entrenamiento. Un gamma bajo significa 'lejos' y un gamma alto significa 'cerca'.

Para los kernels rbf, poly y sigmoid, gamma controla el ancho del kernel. Un gamma pequeño significa un gran radio de influencia, mientras que un gamma grande significa un pequeño radio de influencia.
Cuando se establece en 'auto', gamma usará 1 / n_features.
kernel (Tipo de Kernel): Este parámetro especifica el tipo de kernel a utilizar en el algoritmo. Transforma los datos de entrada a la forma requerida (dimensión superior) para hacerlos no linealmente separables. Has explorado:

'linear': Utiliza un kernel lineal, adecuado para datos linealmente separables.
'poly' (Polinomial): Utiliza un kernel polinomial. Requiere otro parámetro, degree.
'rbf' (Función de Base Radial o Gaussiana): Una opción popular para relaciones no lineales. A menudo es eficaz y tiene menos hiperparámetros que el kernel polinomial.
degree (Grado del Kernel Polinomial): Este parámetro solo se usa cuando el kernel se establece en 'poly'. Especifica el grado de la función del kernel polinomial. Un grado más alto permite límites de decisión más complejos, pero también puede llevar a un sobreajuste (overfitting).

In [ ]:
from sklearn.svm import SVR
import numpy as np

regressor = SVR()
parameters = {'C': [10],
             'epsilon': [0.01],
             'gamma':['auto'],
             'kernel': ['linear', 'poly'],
             'degree': [3]
             }

from sklearn.model_selection import GridSearchCV
gs = GridSearchCV(regressor, parameters, cv=3, verbose = 10, scoring ='neg_mean_absolute_error', n_jobs=-1)

gs = gs.fit(X_train,y_train)

Fitting 3 folds for each of 2 candidates, totalling 6 fits


In [ ]:
#summarize the results of your GRIDSEARCH
print('***GRIDSEARCH RESULTS***')
print("Best score: %f using %s" % (gs.best_score_, gs.best_params_))
means = gs.cv_results_['mean_test_score']
stds = gs.cv_results_['std_test_score']
params = gs.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

#Returns the coefficient of determination R^2 of the prediction.
#Explained variance score: 1 is perfect prediction
gs.score(X_test, y_test)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.scatter(gs.predict(X_train),gs.predict(X_train)-y_train, c="b", label="training data")
plt.scatter(gs.predict(X_test),gs.predict(X_test)-y_test, c="g", label="hold out data")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.legend(loc="upper left")
plt.hlines(y=0, xmin=-10, xmax=50, color="r")
plt.xlim([-10,50])
plt.show()

In [ ]:
from sklearn import metrics

print("MAE train: ", metrics.mean_absolute_error(y_train, gs.predict(X_train)))
print("MSE train: ",metrics.mean_squared_error(y_train, gs.predict(X_train)))
print("RMSE train: ",np.sqrt(metrics.mean_squared_error(y_train, gs.predict(X_train))))
print("r2: ",np.sqrt(metrics.r2_score(y_train, gs.predict(X_train))))

print("MAE test: ", metrics.mean_absolute_error(y_test, gs.predict(X_test)))
print("MSE test: ",metrics.mean_squared_error(y_test, gs.predict(X_test)))
print("RMSE test: ",np.sqrt(metrics.mean_squared_error(y_test, gs.predict(X_test))))
print("r2: ",np.sqrt(metrics.r2_score(y_test, gs.predict(X_test))))